# 10 · Run an experiment queue (Phase 4 onwards)

**Runtime → Change runtime type → T4 GPU**, then *Run all*. Rerun this notebook in later sessions:
finished samples are on Google Drive and are skipped, so every session continues where the last
one stopped.

Default queue: `configs/jobs/phase4_reproduction.yaml`
1. Table 1 main row: official code, 15 images × 50 seeds.
2. Our re-implementation on 5 seeds (a GPU-side equivalence check).
3. The T = 40, η = 1 variant (20 seeds).
4. The ANN k = 5 variant (20 seeds).

Each session stops cleanly before `BUDGET_MIN` minutes. Close the runtime afterwards
(**Runtime → Disconnect and delete runtime**).

In [1]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK   = "10_run_queue"
JOB_FILE   = "configs/jobs/phase4_reproduction.yaml"
BUDGET_MIN = 100      # this session's GPU minutes; the queue stops before exceeding it
ONLY       = ""       # optional: run just one job by name, e.g. "official_paper_fp32"

In [2]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU (then rerun)."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| bf16 native:", torch.cuda.get_device_capability(0) >= (8, 0))

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07

torch 2.14.1+cu130 | CUDA 13.0 | bf16 native: False


In [3]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

Mounted at /content/drive
project code at /content/project -> ['.git', '.gitignore', '2606.04299v1.pdf', 'PROJECT_PLAN.md', 'README.md', 'configs', 'data', 'docs', 'env', 'notebooks', 'patches', 'pyproject.toml', 'scripts', 'src', 'tests']


In [4]:
SID_ROOT = "/content/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)
subprocess.run(["git", "-C", SID_ROOT, "checkout", "-q", SID_COMMIT], check=True)
!pip install -q -r {PROJECT}/env/requirements-colab.txt
!pip install -q --no-deps -e {SID_ROOT}
!pip install -q -e {PROJECT}
os.environ["SID_ROOT"] = SID_ROOT
# editable installs are only picked up by *new* processes; make them importable in this kernel too
for p in (SID_ROOT, f"{PROJECT}/src"):
    if p not in sys.path:
        sys.path.insert(0, p)
import efficient_sid, sidx
print("efficient_sid OK | sidx", sidx.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for efficient-sid (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sidx (pyproject.toml) ... done
efficient_sid OK | sidx 0.1.0


## Data: fetch the images and verify them against the committed manifests (sha256)

In [5]:
r = subprocess.run([sys.executable, "scripts/prepare_natural.py", "--verify", "--sets", "uncond/sinddm15",
                    "--sid-root", SID_ROOT], cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
assert r.returncode == 0, "data check failed (see above)"

uncond/sinddm15: 15 images verified against data/manifests/uncond_sinddm15.csv
 


## What is still pending

In [6]:
OUT_ROOT = f"{DRIVE_ROOT}/outputs"
base = [sys.executable, "scripts/run_queue.py", JOB_FILE, "--out-root", OUT_ROOT, "--var", f"SID_ROOT={SID_ROOT}"]
if ONLY:
    base += ["--only", ONLY]
print(subprocess.run(base + ["--dry-run"], cwd=PROJECT, capture_output=True, text=True).stdout)

official_paper_fp32            sid_official     hash=0bc79975 images=15 seeds=50 pending=750
core_paper_fp32                sid_core         hash=3e7b0bea images=15 seeds=5 pending=75
official_T40_eta1_fp32         sid_official     hash=8fd9ff38 images=15 seeds=20 pending=300
official_ann_k5_fp32           sid_official     hash=ec00840a images=15 seeds=20 pending=300



## Run (stops cleanly before the budget; output streams below)

In [7]:
ret = subprocess.run(base + ["--budget-min", str(BUDGET_MIN)], cwd=PROJECT).returncode
print("runner exit code", ret)
print(subprocess.run(base + ["--dry-run"], cwd=PROJECT, capture_output=True, text=True).stdout)

KeyboardInterrupt: 

In [8]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")

saved /content/drive/MyDrive/CS787/env_logs/2026-11-13_1827_10_run_queue_freeze.txt
